In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [ ]:
# Import databases / data

In [ ]:
conn = sqlite3.connect('customer_churn.db')

sql_query = """
SELECT name
FROM sqlite_master
WHERE type='table';

"""

tables = pd.read_sql(sql_query, conn)

# Create  dataframe for each table
for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    globals()[f"df_{table_name}"] = df
    print(f"Created dataframe: df_{table_name}")

conn.close()    

In [ ]:
#Print table names and column names
conn = sqlite3.connect('customer_churn.db')

for table_name in tables['name']:
    print(f"\nTable Name: {table_name}")
    # Get column information 
    columns_query = f"PRAGMA table_info({table_name});"
    columns = pd.read_sql(columns_query, conn)
    print("columns:")
    print(columns['name'].tolist())

# Close connection    
conn.close()

In [ ]:
#2. Data cleaning 

In [ ]:
df_db_customer.head()

In [ ]:
df_db_customer.info()

In [ ]:
# a. remove col - name
# b. drop columns - interests and pincode
# c. change data type - dob
# d. data standardization - gender
# e. fix missing values - country

In [ ]:
# a. remove col - name

df_db_customer.rename(columns = { 'name':'customer_name'}, inplace = True )
df_db_customer

In [ ]:
df_db_customer

In [ ]:
df_db_customer = df_db_customer.drop(['interests','pincode'],axis = 1)

In [ ]:
print(type(df_db_customer))
df_db_customer.head()

In [ ]:
print(df_db_customer.columns)

In [ ]:
print(df_db_customer.columns.tolist())

In [ ]:
#c. change data type - dob 

df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])

In [ ]:
# d. data standardization - gender

#df_db_customer['gender'].unique()
df_db_customer['gender'] = df_db_customer['gender'].replace({'Men':'Male', 'Women':'Female'})

In [ ]:
df_db_customer['gender']

In [ ]:
# e. fix missing values - country

df_db_customer[df_db_customer['country'].isna()]
    

In [ ]:
# country and state -unique value pair

state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()
df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))

In [ ]:
df_db_customer[df_db_customer['country'].isna()]

In [ ]:
df_db_customer[['country','state']]

In [ ]:
# Cleaning table 2

In [ ]:
df_db_subscription.head()

In [ ]:
df_db_subscription.info()

In [ ]:
date_col = ['subscription_start_date','renewal_date', 'cancellation_date']

df_db_subscription[date_col] = df_db_subscription[date_col].apply(pd.to_datetime)
df_db_subscription.info()

In [ ]:
df_db_subscription.shape

In [ ]:
# Cleaning

In [ ]:
df_db_support.head()

In [ ]:
df_db_support.info()

In [ ]:
df_db_support.drop(columns = ['col_1','comment'],inplace = True)

In [ ]:
df_db_support.info()

In [ ]:
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])

In [ ]:
df_db_support.info()

In [ ]:
df_db_support.shape

In [ ]:
# 3. Feature Engineering and Data Analysis

In [ ]:
# Create a new col using existing col - churn flag

In [ ]:
df_db_subscription['churn_flag']= np.where(df_db_subscription['cancellation_date'].notna(), 1,0)

In [ ]:
df_db_subscription.head()

In [ ]:
df = (df_db_subscription
            .merge(df_db_customer, on= 'customerid', how= 'left')
            .merge(df_db_support, on= 'customerid', how= 'left'))

In [ ]:
df_db_customer.shape

In [ ]:
df_db_subscription.shape

In [ ]:
df_db_support.shape

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df_db_subscription['customerid'].nunique()

In [ ]:
df_db_customer['customerid'].nunique()

In [ ]:
df_db_support['customerid'].nunique()

In [ ]:
df_db_support['customerid'].size

In [ ]:
df_db_support

In [ ]:
df_db_support['complaint_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')

In [ ]:
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customerid', keep = 'last')

In [ ]:
df_db_support['customerid'].size

In [ ]:
# merge df
df = (df_db_subscription
                  .merge(df_db_customer, on= 'customerid', how= 'left')
                  .merge(df_db_support, on = 'customerid', how= 'left'))

In [ ]:
df.shape

In [ ]:
df.to_csv('exported_churn_data.csv',index=False)

In [ ]:
## Data Analysis

In [ ]:
# 1. Churn Rate

In [ ]:
df.columns

In [ ]:
churn_rate = df['churn_flag'].mean()*100
print("churn Rate = ", round(churn_rate ,2), "%")

In [ ]:
#  Retention Rate
retention_rate = 100 - churn_rate
print("Retention Rate = ", round(retention_rate,2), "%")

In [ ]:
df.head(2)

In [ ]:
# churn by Plan type
churn_by_plan = (df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name='churn_rate_pct'))

print(churn_by_plan)

In [ ]:
# 4a. churn by state + sum(revenure) &  count of users 
churn_by_state = (
    df.groupby('state')
      .agg(
          churn_rate_pct=('churn_flag', lambda x: round(x.mean() * 100, 2)),
          total_revenue=('monthly_charges', 'sum'),
          total_users=('customerid', 'count')
      )
      .reset_index()
)

print(churn_by_state)

In [ ]:
# 4b. churn by subscription type + sum(revenue) & count of users
churn_by_subscription = (
    df.groupby('subscription_type')
       .agg(
           churn_rate_pct=('churn_flag', lambda x: round(x.mean() *100,2)),
           total_revenue=('monthly_charges','sum'),
           total_users=('customerid', 'count')
       )
        .reset_index()
)

print(churn_by_subscription)

In [ ]:
# 5.  ARPU - Avg Revenue per user
arpu = df['monthly_charges'].mean()
print('ARPU =', round(arpu,2))

In [ ]:
# 6. Average customer tenure
# count of days users used our service : cancellation date else current date
today = pd.Timestamp.today()

df['tenure_days'] = np.where(
      df['cancellation_date'].notna(),

    (df['cancellation_date'] - df['subscription_start_date']).dt.days,

    (today - df['subscription_start_date']).dt.days
)

avg_tenure = df['tenure_days'].mean()
print("Avg Tenure ( Days) = ", round(avg_tenure),0)

In [ ]:
#7. Revenue at risk - revenue last from churned users
revenue_at_risk = df.loc[df['churn_flag']==1, 'monthly_charges'].sum()
print("Revenue at Risk (Rs 'K')=", revenue_at_risk)

In [ ]:
#8.Escalation Rate
escalation_rate = (df['escalations']=='Y').mean()*100
print("Escalation Rate = ", round(escalation_rate, 2), "%")

In [ ]:
#9. Average complaint per users
avg_complaints = df['complaint_count'].sum() / df['customerid'].nunique()
print("Avg_complaints per users = ", round(avg_complaints ,2))

In [ ]:
# 10. Correlation Escalation vs churn

#df['escalation'] = np.where(df['escalations'] == 'Y' , 1, 0) # encoding string to int type
corr_df = df[['escalation','churn_flag']].dropna()

correlation = corr_df['escalation'].corr(df['churn_flag'])

print("Correlation between escalation vs churn is = ", round(correlation,2))

In [ ]:
df['escalations'] = np.where(df['escalation'] == 'Y', 1, 0)

In [ ]:
df['escalations']

In [ ]:
# 11. churn risk - create a column using existing col
conditions = [
    (df['churn_score'] < 50),
    (df['churn_score'] >= 50) & (df['churn_score'] < 70),
    (df['churn_score'] >= 70)]

choices = ['low','med','high']

df['churn_risk'] = np.select(conditions, choices, default='unknown')

In [ ]:
df[['churn_risk','churn_score']].head

In [ ]:
# 4. Visualization using Matplotlib

In [ ]:
df_visual = df.copy()

In [ ]:
df_visual.shape

In [ ]:
df_visual.columns

In [ ]:
# 4.1 Monthly churn Trend (Time series KPI)

df_visual['cancellation_month'] = df_visual['cancellation_date'].dt.to_period('M')

churn_trend = df_visual[df_visual['churn_flag'] == 1].groupby('cancellation_month').size()

plt.figure(figsize=(8,3))
plt.plot(churn_trend.index.astype(str), churn_trend.values, color='green', marker='o', linestyle='dashed', linewidth=2, markersize=12)   

plt.title('Monthly Churn Trend')
plt.xlabel('Month')
plt.ylabel('Churned Customers')
plt.show()

In [ ]:
# 4.2 Churn by Plan type
churn_plan = df_visual.groupby('plan_type')['churn_flag'].mean()

#colors = ['yellow','green','blue']
colors = plt.cm.Set2(np.linspace(0,1, len(churn_plan)))

plt.figure(figsize=(7,4))

plt.bar(churn_plan.index, churn_plan.values, color = colors)
plt.show

In [ ]:
# 4.2 Churn by States
churn_plan = df_visual.groupby('state')['churn_flag'].mean()

#colors = ['yellow','green','blue']
colors = plt.cm.Set2(np.linspace(0,1, len(churn_plan)))

plt.figure(figsize=(12,4))

plt.bar(churn_plan.index, churn_plan.values, color = colors)
plt.show

In [ ]:
# Visualization Using Seaborn

In [ ]:
# encoding - convert str to numeric so that we can find corr between features
df_visual.columns

In [ ]:
# df_encoded =
df_visual[['plan_type','contract_type','churn_score','churn_flag','churn_risk','escalation']].head()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# incorrect method of encoding - as numbers are not assigned based on priority

df_encoded = df_visual[['plan_type','contract_type','churn_score','churn_flag','churn_risk','escalation']]

categorial_cols = ['plan_type','contract_type','churn_risk',]

for col in categorial_cols:
    df_encoded[col] = df_encoded[col].astype('category').cat.codes

In [ ]:
df_encoded.head()

In [ ]:
# Heatmap (correlation matrix)

sns.heatmap(df_encoded.corr(), annot=True)

In [ ]:
df_visual['churn_risk'].unique()

In [ ]:
# Correct methods of encoding - based on priority
df_encoded = df_visual[['plan_type','contract_type','churn_score','churn_flag','churn_risk','escalation']]

order_mappings = {
    'plan_type' : ['Basic','Standard','Premium'],
    'contract_type' : ['Monthly','Annual'],
    'churn_risk' : ['low','med','high']
    }
    
for col, order in order_mappings.items():
    df_encoded[col] = pd.Categorical(df_encoded[col].astype('category'), categories=order, ordered=True).codes
    
     
    
    

In [ ]:
df_encoded.head()

In [ ]:
# Heatmap (correlation matrix)

sns.heatmap(df_encoded.corr(), annot=True)

In [ ]:
#pairplot - relationship in a dataset
sns.pairplot(df_encoded)

In [ ]:
#catplt/Facegrid plot - multi-dim comparison

sns.catplot(
    data=df_visual,
    x='plan_type',
    y='monthly_charges',
    hue='gender',
    col='churn_risk')

In [ ]:
# Pivot table

In [ ]:
pd.pivot_table(
    df_visual,
    values = 'churn_flag',
    index = 'plan_type',
    aggfunc = 'mean'
).reset_index()    
    

In [ ]:
pd.pivot_table(
    df_visual,
    index = 'plan_type',
    values = ['monthly_charges','customerid','churn_flag'],
    aggfunc = {
        'monthly_charges': 'sum' , 
        'customerid': 'nunique',
        'churn_flag': 'mean'
    }    
        
)

In [ ]:
# working with sql in python (pandas)

In [ ]:
# Create db in sql
conn = sqlite3.connect('test_database.sqlite')

#table details
conn.execute("CREATE TABLE users (first_names TEXT, country TEXT, budget INTEGER)")

# commit and save
conn.commit()

In [ ]:
#Insert data
cursor = conn.cursor()

# Write SQL query to insert records into SQL table
cursor.execute(
    """
        INSERT INTO users VALUES
           ('Madhav', 'India', 5000),
           ('Rishabh', 'India', 2500),
           ('Vishakha', 'India', 3500)
   """
)        
        
# Commit and save
conn.commit()

print("Data Inserted successfully")

In [ ]:
# Check inserted data in table
conn = sqlite3.connect('test_database.sqlite')
query = """SELECT * FROM users"""

df_results = pd.read_sql(query,conn)

df_results.head()

In [ ]:
# aggregation
query ="""
       SELECT country, SUM(budget) as total_budget
       FROM users
       GROUP BY country
"""
df_agg = pd.read_sql(query, conn)
df_agg

In [ ]:
# Always close the conn with db once the task is over 
conn.close()